# Exercise 2

Sampling from discrete distributions. Mostly just trying the methods and checking they match the target probs.

## Imports and setup

Usual imports. Nothing fancy, just `random`, `math`, `time`, `statistics`, and plotting if its there.

In [ ]:
import math
import os
import random
import statistics
import time

try:
    import numpy as np
except ImportError:
    np = None

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from scipy import stats
except ImportError:
    stats = None

PLOT_DIR = "Exercise2/pics"
SEED = 20240615

random.seed(SEED)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, filename):
    if plt is None:
        print("matplotlib not available, skipping", filename)
        return
    ensure_dir(PLOT_DIR)
    out_path = os.path.join(PLOT_DIR, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print("saved", out_path)


ensure_dir(PLOT_DIR)
print("seed =", SEED)
print("numpy available:", np is not None)
print("scipy available:", stats is not None)
print("matplotlib available:", plt is not None)


## Helper functions

Small helper stuff for counts, chi-square, TV distance, and the plots.

In [ ]:
def draw_u():
    return random.random()


def observed_counts(values, categories):
    counts = {x: 0 for x in categories}
    for value in values:
        counts[value] += 1
    return counts


def total_variation_distance(empirical, theoretical):
    return 0.5 * sum(abs(empirical[x] - theoretical[x]) for x in theoretical)


def chi_square_test_counts(counts, expected_probs, n, df_adjust=0):
    keys = list(expected_probs.keys())
    statistic = 0.0
    for key in keys:
        expected = n * expected_probs[key]
        observed = counts[key]
        statistic += (observed - expected) ** 2 / expected
    df = len(keys) - 1 - df_adjust
    pvalue = float(stats.chi2.sf(statistic, df)) if stats is not None else None
    return {
        "statistic": statistic,
        "df": df,
        "pvalue": pvalue,
        "reject_5pct": (pvalue < 0.05) if pvalue is not None else None,
    }


def print_rows(rows, headers):
    widths = []
    for i, header in enumerate(headers):
        widths.append(max(len(header), max(len(str(row[i])) for row in rows)))
    print(" | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))


def plot_probs_comparison(labels, observed, theoretical, title, filename):
    if plt is None:
        return
    x = range(len(labels))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar([i - 0.2 for i in x], observed, width=0.4, label="observed", color="steelblue")
    ax.bar([i + 0.2 for i in x], theoretical, width=0.4, label="theoretical", color="orange", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_ylabel("probability")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    save_fig(fig, filename)
    plt.close(fig)


## Part 1 - Geometric

Using

\[
P(X=n) = (1-p)^{n-1}p
\]

and inverse transform

\[
X = \left\lfloor \frac{\log(U)}{\log(1-p)} \right\rfloor + 1
\]

which is the usual way to do it.

In [ ]:
def sample_geometric_inverse(p):
    u = draw_u()
    return math.floor(math.log(u) / math.log(1 - p)) + 1


def geometric_pmf(x, p):
    return ((1 - p) ** (x - 1)) * p


def geometric_grouped_probs(p, K):
    probs = {}
    for x in range(1, K + 1):
        probs[str(x)] = geometric_pmf(x, p)
    probs[f">{K}"] = 1 - sum(probs.values())
    return probs


def group_geometric_sample(values, K):
    grouped = {str(x): 0 for x in range(1, K + 1)}
    grouped[f">{K}"] = 0
    for x in values:
        if x <= K:
            grouped[str(x)] += 1
        else:
            grouped[f">{K}"] += 1
    return grouped


def analyze_geometric(p, n, K, filename):
    values = [sample_geometric_inverse(p) for _ in range(n)]
    sample_mean = statistics.mean(values)
    sample_variance = statistics.pvariance(values)
    theo_mean = 1 / p
    theo_variance = (1 - p) / (p ** 2)

    grouped_counts = group_geometric_sample(values, K)
    grouped_probs = geometric_grouped_probs(p, K)
    chi = chi_square_test_counts(grouped_counts, grouped_probs, n)

    empirical_probs = {key: grouped_counts[key] / n for key in grouped_counts}
    tv = total_variation_distance(empirical_probs, grouped_probs)

    labels = list(grouped_probs.keys())
    observed = [empirical_probs[label] for label in labels]
    theoretical = [grouped_probs[label] for label in labels]
    plot_probs_comparison(labels, observed, theoretical, f"Geometric distribution, p={p}", filename)

    return {
        "p": p,
        "values": values,
        "sample_mean": sample_mean,
        "theoretical_mean": theo_mean,
        "sample_variance": sample_variance,
        "theoretical_variance": theo_variance,
        "grouped_counts": grouped_counts,
        "grouped_probs": grouped_probs,
        "chi": chi,
        "tv": tv,
    }


In [ ]:
geom_n = 10_000
geom_settings = [
    (0.1, 20, "geometric_p_0_1.png"),
    (0.4, 10, "geometric_p_0_4.png"),
    (0.8, 6, "geometric_p_0_8.png"),
]

geom_results = []
for p, K, filename in geom_settings:
    result = analyze_geometric(p, geom_n, K, filename)
    geom_results.append(result)
    print("\np =", p)
    print("sample mean:", round(result["sample_mean"], 4))
    print("theoretical mean:", round(result["theoretical_mean"], 4))
    print("sample variance:", round(result["sample_variance"], 4))
    print("theoretical variance:", round(result["theoretical_variance"], 4))
    print("chi-square:", result["chi"])
    print("total variation distance:", round(result["tv"], 5))


In [ ]:
geom_rows = []
for result in geom_results:
    geom_rows.append([
        result["p"],
        round(result["sample_mean"], 4),
        round(result["theoretical_mean"], 4),
        round(result["sample_variance"], 4),
        round(result["theoretical_variance"], 4),
        round(result["chi"]["statistic"], 4),
        result["chi"]["df"],
        None if result["chi"]["pvalue"] is None else round(result["chi"]["pvalue"], 4),
        round(result["tv"], 5),
    ])

print("Geometric summary table")
print_rows(
    geom_rows,
    [
        "p",
        "sample mean",
        "theory mean",
        "sample var",
        "theory var",
        "chi-square",
        "df",
        "p-value",
        "TV distance",
    ],
)


This part came out about how I expected. Small `p` gives a longer tail and bigger variance, while large `p` shoves most values down near 1.

## Part 2 - Six-point distribution

Now its the fixed six-point one from the exercise.

| x | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| P(X=x) | 7/48 | 5/48 | 1/8 | 1/16 | 1/4 | 5/16 |

In [ ]:
values = [1, 2, 3, 4, 5, 6]
probs = [7/48, 5/48, 1/8, 1/16, 1/4, 5/16]
prob_dict = dict(zip(values, probs))

print("sum of probabilities =", sum(probs))
print("theoretical probabilities:", prob_dict)


In [ ]:
def build_cumulative(probs):
    cumulative = []
    running = 0.0
    for p in probs:
        running += p
        cumulative.append(running)
    return cumulative


cum_probs = build_cumulative(probs)


def sample_direct_once(values, cumulative):
    u = draw_u()
    for value, cutoff in zip(values, cumulative):
        if u <= cutoff:
            return value
    return values[-1]


def sample_direct(values, probs, n):
    cumulative = build_cumulative(probs)
    return [sample_direct_once(values, cumulative) for _ in range(n)]


def sample_rejection(values, probs, n):
    c = max(probs)
    accepted = []
    proposals = 0
    while len(accepted) < n:
        proposals += 1
        idx = random.randrange(len(values))
        u = draw_u()
        if u <= probs[idx] / c:
            accepted.append(values[idx])
    acceptance_rate = n / proposals
    return accepted, {
        "c": c,
        "proposals": proposals,
        "acceptance_rate": acceptance_rate,
        "avg_proposals_per_sample": proposals / n,
        "theoretical_acceptance_rate": 1 / (len(values) * c),
    }


def build_alias_table(probs):
    k = len(probs)
    scaled = [p * k for p in probs]
    small = []
    large = []
    prob_table = [0.0] * k
    alias_table = [0] * k

    for i, x in enumerate(scaled):
        if x < 1:
            small.append(i)
        else:
            large.append(i)

    while small and large:
        s = small.pop()
        l = large.pop()
        prob_table[s] = scaled[s]
        alias_table[s] = l
        scaled[l] = scaled[l] - (1 - scaled[s])
        if scaled[l] < 1:
            small.append(l)
        else:
            large.append(l)

    for i in small + large:
        prob_table[i] = 1.0
        alias_table[i] = i

    return prob_table, alias_table


def sample_alias(values, probs, n):
    prob_table, alias_table = build_alias_table(probs)
    out = []
    k = len(values)
    for _ in range(n):
        col = random.randrange(k)
        u = draw_u()
        if u < prob_table[col]:
            out.append(values[col])
        else:
            out.append(values[alias_table[col]])
    return out, {"prob_table": prob_table, "alias_table": alias_table}


In [ ]:
six_n = 100_000


def analyze_sixpoint_method(method_name, samples, filename, extra=None):
    counts = observed_counts(samples, values)
    empirical = {x: counts[x] / len(samples) for x in values}
    chi = chi_square_test_counts(counts, prob_dict, len(samples))
    tv = total_variation_distance(empirical, prob_dict)
    plot_probs_comparison(
        [str(x) for x in values],
        [empirical[x] for x in values],
        [prob_dict[x] for x in values],
        f"Six-point distribution - {method_name}",
        filename,
    )
    return {
        "method": method_name,
        "counts": counts,
        "empirical": empirical,
        "chi": chi,
        "tv": tv,
        "extra": extra or {},
    }


start = time.perf_counter()
direct_samples = sample_direct(values, probs, six_n)
direct_time = time.perf_counter() - start
direct_result = analyze_sixpoint_method("Direct", direct_samples, "sixpoint_direct.png")
direct_result["runtime"] = direct_time

start = time.perf_counter()
rejection_samples, rejection_info = sample_rejection(values, probs, six_n)
rejection_time = time.perf_counter() - start
rejection_result = analyze_sixpoint_method("Rejection", rejection_samples, "sixpoint_rejection.png", rejection_info)
rejection_result["runtime"] = rejection_time

alias_setup_start = time.perf_counter()
alias_prob_table, alias_table = build_alias_table(probs)
alias_setup_time = time.perf_counter() - alias_setup_start

start = time.perf_counter()
alias_samples, alias_info = sample_alias(values, probs, six_n)
alias_sample_time = time.perf_counter() - start
alias_result = analyze_sixpoint_method("Alias", alias_samples, "sixpoint_alias.png", alias_info)
alias_result["runtime"] = alias_setup_time + alias_sample_time
alias_result["setup_time"] = alias_setup_time
alias_result["sample_time"] = alias_sample_time

print("alias prob table:", [round(x, 4) for x in alias_prob_table])
print("alias table:", alias_table)


In [ ]:
six_rows = []
for result in [direct_result, rejection_result, alias_result]:
    extra = result["extra"]
    six_rows.append([
        result["method"],
        round(result["runtime"], 5),
        round(result["chi"]["statistic"], 4),
        None if result["chi"]["pvalue"] is None else round(result["chi"]["pvalue"], 4),
        round(result["tv"], 5),
        "-" if "acceptance_rate" not in extra else round(extra["acceptance_rate"], 4),
        "-" if "avg_proposals_per_sample" not in extra else round(extra["avg_proposals_per_sample"], 4),
    ])

print("Six-point comparison table")
print_rows(
    six_rows,
    [
        "method",
        "runtime",
        "chi-square",
        "p-value",
        "TV distance",
        "accept rate",
        "avg props/sample",
    ],
)

print("\nRejection info:")
print(rejection_result["extra"])


All 3 methods should land on the same target distribution if coded right. If one p-value is a bit low once, thats not a disaster by itself.

## Part 3 - Performance

Here I just timed them for a few sample sizes. For alias I split setup and sampling bc otherwise its a bit misleading.

In [ ]:
sizes = [1_000, 10_000, 100_000, 500_000]
runtime_rows = []
runtime_data = {
    "n": [],
    "direct": [],
    "rejection": [],
    "alias_total": [],
    "alias_setup": [],
    "alias_sampling": [],
}

for n in sizes:
    start = time.perf_counter()
    direct_samples = sample_direct(values, probs, n)
    direct_t = time.perf_counter() - start

    start = time.perf_counter()
    rejection_samples, rej_info = sample_rejection(values, probs, n)
    rejection_t = time.perf_counter() - start

    alias_setup_start = time.perf_counter()
    alias_prob_table, alias_table = build_alias_table(probs)
    alias_setup_t = time.perf_counter() - alias_setup_start

    start = time.perf_counter()
    alias_samples = []
    k = len(values)
    for _ in range(n):
        col = random.randrange(k)
        u = draw_u()
        if u < alias_prob_table[col]:
            alias_samples.append(values[col])
        else:
            alias_samples.append(values[alias_table[col]])
    alias_sampling_t = time.perf_counter() - start
    alias_total_t = alias_setup_t + alias_sampling_t

    direct_emp = {x: c / n for x, c in observed_counts(direct_samples, values).items()}
    rejection_emp = {x: c / n for x, c in observed_counts(rejection_samples, values).items()}
    alias_emp = {x: c / n for x, c in observed_counts(alias_samples, values).items()}

    runtime_rows.append([
        n,
        round(direct_t, 5),
        round(rejection_t, 5),
        round(alias_setup_t, 5),
        round(alias_sampling_t, 5),
        round(alias_total_t, 5),
        round(total_variation_distance(direct_emp, prob_dict), 5),
        round(total_variation_distance(rejection_emp, prob_dict), 5),
        round(total_variation_distance(alias_emp, prob_dict), 5),
    ])

    runtime_data["n"].append(n)
    runtime_data["direct"].append(direct_t)
    runtime_data["rejection"].append(rejection_t)
    runtime_data["alias_total"].append(alias_total_t)
    runtime_data["alias_setup"].append(alias_setup_t)
    runtime_data["alias_sampling"].append(alias_sampling_t)

print_rows(
    runtime_rows,
    [
        "n",
        "direct",
        "rejection",
        "alias setup",
        "alias sampling",
        "alias total",
        "TV direct",
        "TV reject",
        "TV alias",
    ],
)


In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(runtime_data["n"], runtime_data["direct"], marker="o", label="direct")
    ax.plot(runtime_data["n"], runtime_data["rejection"], marker="o", label="rejection")
    ax.plot(runtime_data["n"], runtime_data["alias_total"], marker="o", label="alias total")
    ax.set_xlabel("sample size")
    ax.set_ylabel("runtime (seconds)")
    ax.set_title("Runtime comparison")
    ax.legend()
    ax.grid(alpha=0.2)
    save_fig(fig, "method_runtime_comparison.png")
    plt.close(fig)


In [ ]:
if plt is not None:
    labels = ["direct", "rejection", "alias"]
    tv_values = [direct_result["tv"], rejection_result["tv"], alias_result["tv"]]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(labels, tv_values, color=["steelblue", "darkorange", "seagreen"])
    ax.set_ylabel("total variation distance")
    ax.set_title("Accuracy comparison at n=100000")
    ax.grid(axis="y", alpha=0.2)
    save_fig(fig, "method_accuracy_comparison.png")
    plt.close(fig)


Pretty standard result really. Direct is easiest and totally fine when `k=6`. Rejection is simple too but wastes draws. Alias is more setup work but makes sense when you sample a lot from the same probs.

## Part 4 - What I would use

**Direct/crude**

- easiest one
- good for small support
- gets slower if there are loads of categories and I keep linear search

**Rejection**

- also simple
- can waste a lot if accept rate is bad
- here its ok but not the nicest

**Alias**

- more annoying to set up
- fast after that
- good when the same discrete law is used many times

## End

For the geometric part the inverse method worked fine. For the six-point part all 3 methods matched pretty well. If I only had `k=6` I would honestly just use the direct method. Alias only starts feeling worth it when there are many repeated draws.